# inf-masking — ex2: pad-mask attention — drop PAD tokens from the keys

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `inf-masking`. Running the final beacon cell reports progress against the `Numpy: Inf-fill masking trick` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Inf-fill masking trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inf-masking`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inf-masking"
DD_SUBTOPIC = "Numpy: Inf-fill masking trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## inf-fill masking before softmax — quick refresher

Replacing forbidden positions with `-inf` before softmax sends their weight to exactly zero (since `exp(-inf) = 0`) while leaving the unmasked positions to renormalise to one. The trick is shape-agnostic:

```python
masked = scores.masked_fill(forbidden, float('-inf'))
weights = masked.softmax(dim=-1)
```

**This drill (ex2) vs ex1.** ex1 applied a **causal** mask — a lower-triangular pattern that forbids future positions, the same for every batch element. ex2 applies a **pad** mask — a *per-batch* boolean over the key axis that forbids PAD tokens. Same `-inf` trick, different mask topology (one is fixed per-sequence-length, the other varies per-batch-element).

### Exercise 2 — pad-mask attention — drop PAD tokens from the keys

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `masked_fill(-inf)` with a per-batch `(B, S_k)` pad mask broadcast across the query axis to a `(B, S_q, S_k)` attention scores tensor, producing softmax weights that place zero mass on PAD-token keys.
> Keywords: pad-mask, attention, masked_fill, per-batch-mask
> ```

**KCs targeted:** `masked-fill-neg-inf-before-softmax`, `pad-mask-broadcast-on-keys`

Implement `ex2_pad_mask_attention(scores, pad_mask)`.

You have raw attention scores `scores` of shape `(B, S_q, S_k)` and a key-side pad mask `pad_mask` of shape `(B, S_k)`, where `pad_mask[b, j] = True` iff key position `j` in batch element `b` is a PAD token that must NOT receive attention.

**Rules.**
1. Broadcast `pad_mask` to `(B, 1, S_k)` so it applies to every query row but is independent for each batch element.
2. Use `masked_fill` with `float('-inf')` to set forbidden positions.
3. Softmax along `dim=-1` (the key axis).
4. After softmax, the weight on any masked key must be EXACTLY 0 (not just small). Each row over un-masked keys must sum to 1.0.

Inputs:
- `scores`: `(B, S_q, S_k)` float tensor.
- `pad_mask`: `(B, S_k)` bool tensor, `True` = PAD = forbidden.

Output: `(B, S_q, S_k)` softmax weights with zero mass on PAD keys.

The visualization renders the post-softmax attention weights as a heatmap per batch element so you can see the masked columns disappear.

In [ ]:
def ex2_pad_mask_attention(scores: Tensor, pad_mask: Tensor) -> Tensor:
    """Pad-masked softmax: zero attention weight on PAD-token keys."""
    raise NotImplementedError()


def _test_ex2():
    # Hand-built scores so the masked weights are predictable.
    scores = t.tensor([
        # batch 0: last two keys are PAD
        [[0.0, 0.0, 5.0, 5.0],
         [1.0, 1.0, 0.0, 0.0],
         [2.0, 0.0, 0.0, 0.0]],
        # batch 1: only key 0 is valid (extreme)
        [[3.0, 0.0, 0.0, 0.0],
         [4.0, 9.9, 9.9, 9.9],
         [0.0, 9.9, 9.9, 9.9]],
    ])
    pad_mask = t.tensor([
        [False, False, True,  True],
        [False, True,  True,  True],
    ])
    w = ex2_pad_mask_attention(scores, pad_mask)
    assert w.shape == scores.shape
    assert w.dtype == t.float32

    # Masked positions must be exactly zero.
    assert t.all(w[0, :, 2:] == 0), f'b=0 PAD columns not zero:\n{w[0]}'
    assert t.all(w[1, :, 1:] == 0), f'b=1 PAD columns not zero:\n{w[1]}'

    # Rows sum to 1 — only over un-masked keys, but softmax renormalises.
    row_sums = w.sum(dim=-1)
    assert t.allclose(row_sums, t.ones_like(row_sums), atol=1e-6), f'row sums:\n{row_sums}'

    # Batch 1: only key 0 is valid → its weight must be 1.0 for every query.
    assert t.allclose(w[1, :, 0], t.ones(3), atol=1e-6), f'b=1 single-valid-key not 1.0: {w[1, :, 0]}'

    # Batch 0 row 0: scores [0, 0] over un-masked keys → uniform [0.5, 0.5].
    assert t.allclose(w[0, 0, :2], t.tensor([0.5, 0.5]), atol=1e-6), f'uniform-over-valid wrong: {w[0,0,:2]}'

    # --- Heatmap visualization ---
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    for b in range(2):
        axes[b].imshow(w[b].numpy(), cmap='magma', vmin=0, vmax=1, aspect='auto')
        axes[b].set_title(f'batch {b} — PAD cols ' + str([j for j, m in enumerate(pad_mask[b].tolist()) if m]))
        axes[b].set_xlabel('key index')
        axes[b].set_ylabel('query index')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_pad_mask_attention(scores: Tensor, pad_mask: Tensor) -> Tensor:
    # (B, S_k) -> (B, 1, S_k) so it broadcasts across queries.
    masked = scores.masked_fill(pad_mask.unsqueeze(1), float('-inf'))
    return masked.softmax(dim=-1)
```

**Why broadcast on `dim=1`.** The pad mask only knows about KEYS (which sequence positions are PAD), not queries. The forbidden set is the same for every query but varies per batch element — so the shape must broadcast `(B, S_k) -> (B, 1, S_k) -> (B, S_q, S_k)`.

**Why `-inf` and not a large negative.** `exp(-1e9) ≈ 0` will work for normal-precision softmax, but `exp(-inf) == 0` exactly. The test checks `== 0` (not `< 1e-9`) precisely to catch the `large-negative-but-not-inf` mistake.

**Difference from ex1.** ex1's causal mask was a `(T, T)` lower-triangular pattern that depended only on positions (same across batch). ex2's pad mask is `(B, S_k)` — every batch element has its OWN PAD pattern, so the mask must broadcast through the batch axis correctly. The `unsqueeze(1)` on the query axis (not the batch axis) is the crucial detail.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()